# 🚀 Huấn luyện YOLO26n — UAV Human Detection & Tracking (Giai đoạn 2: Hợp nhất VisDrone + P-DESTRE)
**Đề tài NCKH:** Human Tracking Drone  
**Mô hình:** YOLO26n (Ultralytics Nano — NMS-free native, non-DFL)  
**Chiến lược bóc tách P-DESTRE chia thành 4 Đợt an toàn:**  
- **Đợt 1:** 16 video Train + 4 video Val (đang chạy / đã xong)  
- **Đợt 2:** 15 video Train + 4 video Val  
- **Đợt 3:** 15 video Train + 4 video Val  
- **Đợt 4:** 14 video Train + 3 video Val (Hoàn tất 100% 75 video P-DESTRE)  
- **Mỗi video đều được checkpoint tức thì vào Google Drive (`pdestre_processed/`).**

### Bước 1: Kết nối Google Drive và Kiểm tra GPU

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Kiểm tra GPU được Colab cấp phát
!nvidia-smi

### Bước 2: Cài đặt Ultralytics & Thư viện Hỗ trợ

In [ ]:
!pip install -q "ultralytics>=8.3.0" opencv-python tqdm

### Bước 3.1: Bóc tách P-DESTRE ĐỢT 1 (Train 1..16 | Val 1..4)
> 🛡️ Đợt 1 gồm 16 video Train + 4 video Val đầu tiên (~2.500 ảnh, ~8-10 phút). Đã checkpoint vào Drive.

In [ ]:
# =====================================================================
# BƯỚC 3.1: BÓC TÁCH P-DESTRE ĐỢT 1 (Train 1..16 | Val 1..4)
# =====================================================================
import os
import tarfile
import cv2
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm
import random

tar_path = "/content/drive/MyDrive/human_tracking_drone/data/dataset.tar"
drive_output_dir = Path("/content/drive/MyDrive/human_tracking_drone/data/pdestre_processed")
drive_output_dir.mkdir(parents=True, exist_ok=True)

temp_extract_dir = Path("/content/temp_extract")
temp_extract_dir.mkdir(parents=True, exist_ok=True)

# 1. Đảm bảo file nhãn annotations đã sẵn sàng (trích xuất ~5 giây nếu chưa có)
ann_dir = Path("/content/pdestre_annotations")
if not ann_dir.exists() or len(list(ann_dir.glob('*.txt'))) == 0:
    print("[*] Đang trích xuất file nhãn annotations từ dataset.tar (rất nhẹ, ~5 giây)...")
    ann_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:*") as tar:
        for m in tar:
            if "annotation" in m.name.lower() and m.name.endswith(".txt") and not os.path.basename(m.name).startswith("._"):
                m.name = os.path.basename(m.name)
                tar.extract(m, ann_dir)
    print(f"[+] Đã trích xuất {len(list(ann_dir.glob('*.txt')))} file nhãn annotations!")
else:
    print(f"[i] Đã có sẵn {len(list(ann_dir.glob('*.txt')))} file nhãn annotations.")

# 2. Phân chia Train (80%) và Val (20%) cố định với seed 42
ann_files = sorted([f.stem for f in ann_dir.glob("*.txt")])
random.seed(42)
shuffled_vids = list(ann_files)
random.shuffle(shuffled_vids)

n_val = max(1, int(len(shuffled_vids) * 0.2))
val_vid_set = set(shuffled_vids[:n_val])
train_vid_set = set(shuffled_vids[n_val:])

train_vids_ordered = [v for v in shuffled_vids if v in train_vid_set]
val_vids_ordered = [v for v in shuffled_vids if v in val_vid_set]

START_TRAIN, END_TRAIN = 0, 16
START_VAL, END_VAL = 0, 4

target_train = set(train_vids_ordered[START_TRAIN:END_TRAIN])
target_val = set(val_vids_ordered[START_VAL:END_VAL])
all_targets = target_train | target_val
all_targets_map = {v.lower(): v for v in all_targets}
ann_file_map = {f.stem.lower(): f for f in ann_dir.glob('*.txt')}

print(f"[*] Mục tiêu ĐỢT 1: {len(target_train)} Train videos ({START_TRAIN+1}..{END_TRAIN}) + {len(target_val)} Val videos ({START_VAL+1}..{END_VAL})")

processed_train = 0
processed_val = 0

print("\n🚀 Bắt đầu Streaming xử lý Video ĐỢT 1...")
with tarfile.open(tar_path, "r:*") as tar:
    for m in tar:
        if not m.name.lower().endswith((".mp4", ".avi", ".mov")) or os.path.basename(m.name).startswith("._"):
            continue
            
        vid_stem = Path(m.name).stem
        if vid_stem.lower() not in all_targets_map:
            continue
            
        actual_vid_stem = all_targets_map[vid_stem.lower()]
        split_type = "train" if actual_vid_stem in target_train else "val"
        zip_drive_file = drive_output_dir / f"{split_type}_{actual_vid_stem}.zip"
        
        # Checkpoint: Bỏ qua nếu video đã có trên Drive
        if zip_drive_file.exists():
            print(f"[i] Checkpoint đã có sẵn: {zip_drive_file.name}, bỏ qua.")
            if split_type == "train": processed_train += 1
            else: processed_val += 1
            continue
            
        print(f"\n[*] [Đợt 1] Đang trích xuất: {m.name} ({m.size / (1024*1024):.1f} MB)...")
        tar.extract(m, temp_extract_dir)
        extracted_file = temp_extract_dir / m.name
        
        if not extracted_file.exists():
            continue
            
        ann_file = ann_file_map.get(actual_vid_stem.lower())
        frame_boxes = {}
        if ann_file and ann_file.exists():
            with open(ann_file, "r", encoding="utf-8") as f:
                for line in f:
                    p = line.strip().split(",")
                    if len(p) >= 6:
                        try:
                            fid = int(p[0])
                            x, y, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
                            if w > 5 and h > 10:
                                frame_boxes.setdefault(fid, []).append((x, y, w, h))
                        except ValueError:
                            continue
                            
        cap = cv2.VideoCapture(str(extracted_file))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        step = int(round(fps))
        
        temp_img_dir = Path(f"/content/tmp_{actual_vid_stem}")
        temp_img_dir.mkdir(parents=True, exist_ok=True)
        
        extracted_frames = 0
        f_idx = 0
        
        while True:
            r, frame = cap.read()
            if not r:
                break
            f_idx += 1
            
            if f_idx % step == 0 and f_idx in frame_boxes:
                h_img, w_img, _ = frame.shape
                frame_base = f"{actual_vid_stem}_f{f_idx:06d}"
                
                frame_resized = cv2.resize(frame, (1920, 1080))
                cv2.imwrite(str(temp_img_dir / f"{frame_base}.jpg"), frame_resized, [cv2.IMWRITE_JPEG_QUALITY, 85])
                
                yolo_lines = []
                for (x, y, w, h) in frame_boxes[f_idx]:
                    xc = max(0.0, min(1.0, (x + w / 2) / w_img))
                    yc = max(0.0, min(1.0, (y + h / 2) / h_img))
                    nw = max(0.0, min(1.0, w / w_img))
                    nh = max(0.0, min(1.0, h / h_img))
                    yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
                
                with open(temp_img_dir / f"{frame_base}.txt", "w", encoding="utf-8") as lf:
                    lf.write("\n".join(yolo_lines))
                
                extracted_frames += 1
                
        cap.release()
        if extracted_file.exists():
            os.remove(extracted_file)
        
        temp_zip = f"/content/{split_type}_{actual_vid_stem}.zip"
        with zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for f in temp_img_dir.glob("*.*"):
                zf.write(f, f.name)
                
        shutil.move(temp_zip, str(zip_drive_file))
        shutil.rmtree(temp_img_dir, ignore_errors=True)
        
        if split_type == "train": processed_train += 1
        else: processed_val += 1
        
        print(f"[+] [ĐỢT 1] ĐÃ LƯU CHECKPOINT DRIVE: {zip_drive_file.name} ({extracted_frames} ảnh)! ({processed_train}/{len(target_train)} Train | {processed_val}/{len(target_val)} Val)")
        
        if processed_train >= len(target_train) and processed_val >= len(target_val):
            print("\n🎉 ĐÃ HOÀN THÀNH TOÀN BỘ MỤC TIÊU ĐỢT 1!")
            break

shutil.rmtree(temp_extract_dir, ignore_errors=True)
print(f"\n🎉 KẾT THÚC ĐỢT 1! Đã bổ sung thành công vào Google Drive!")


### Bước 3.2: Bóc tách P-DESTRE ĐỢT 2 (Train 17..31 | Val 5..8)
> ⚡ Đợt 2 gồm 15 video Train + 4 video Val tiếp theo (~2.500 ảnh mới). Tự động bỏ qua các video Đợt 1.

In [ ]:
# =====================================================================
# BƯỚC 3.2: BÓC TÁCH P-DESTRE ĐỢT 2 (Train 17..31 | Val 5..8)
# =====================================================================
import os
import tarfile
import cv2
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm
import random

tar_path = "/content/drive/MyDrive/human_tracking_drone/data/dataset.tar"
drive_output_dir = Path("/content/drive/MyDrive/human_tracking_drone/data/pdestre_processed")
drive_output_dir.mkdir(parents=True, exist_ok=True)

temp_extract_dir = Path("/content/temp_extract")
temp_extract_dir.mkdir(parents=True, exist_ok=True)

# 1. Đảm bảo file nhãn annotations đã sẵn sàng (trích xuất ~5 giây nếu chưa có)
ann_dir = Path("/content/pdestre_annotations")
if not ann_dir.exists() or len(list(ann_dir.glob('*.txt'))) == 0:
    print("[*] Đang trích xuất file nhãn annotations từ dataset.tar (rất nhẹ, ~5 giây)...")
    ann_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:*") as tar:
        for m in tar:
            if "annotation" in m.name.lower() and m.name.endswith(".txt") and not os.path.basename(m.name).startswith("._"):
                m.name = os.path.basename(m.name)
                tar.extract(m, ann_dir)
    print(f"[+] Đã trích xuất {len(list(ann_dir.glob('*.txt')))} file nhãn annotations!")
else:
    print(f"[i] Đã có sẵn {len(list(ann_dir.glob('*.txt')))} file nhãn annotations.")

# 2. Phân chia Train (80%) và Val (20%) cố định với seed 42
ann_files = sorted([f.stem for f in ann_dir.glob("*.txt")])
random.seed(42)
shuffled_vids = list(ann_files)
random.shuffle(shuffled_vids)

n_val = max(1, int(len(shuffled_vids) * 0.2))
val_vid_set = set(shuffled_vids[:n_val])
train_vid_set = set(shuffled_vids[n_val:])

train_vids_ordered = [v for v in shuffled_vids if v in train_vid_set]
val_vids_ordered = [v for v in shuffled_vids if v in val_vid_set]

START_TRAIN, END_TRAIN = 16, 31
START_VAL, END_VAL = 4, 8

target_train = set(train_vids_ordered[START_TRAIN:END_TRAIN])
target_val = set(val_vids_ordered[START_VAL:END_VAL])
all_targets = target_train | target_val
all_targets_map = {v.lower(): v for v in all_targets}
ann_file_map = {f.stem.lower(): f for f in ann_dir.glob('*.txt')}

print(f"[*] Mục tiêu ĐỢT 2: {len(target_train)} Train videos ({START_TRAIN+1}..{END_TRAIN}) + {len(target_val)} Val videos ({START_VAL+1}..{END_VAL})")

processed_train = 0
processed_val = 0

print("\n🚀 Bắt đầu Streaming xử lý Video ĐỢT 2...")
with tarfile.open(tar_path, "r:*") as tar:
    for m in tar:
        if not m.name.lower().endswith((".mp4", ".avi", ".mov")) or os.path.basename(m.name).startswith("._"):
            continue
            
        vid_stem = Path(m.name).stem
        if vid_stem.lower() not in all_targets_map:
            continue
            
        actual_vid_stem = all_targets_map[vid_stem.lower()]
        split_type = "train" if actual_vid_stem in target_train else "val"
        zip_drive_file = drive_output_dir / f"{split_type}_{actual_vid_stem}.zip"
        
        # Checkpoint: Bỏ qua nếu video đã có trên Drive
        if zip_drive_file.exists():
            print(f"[i] Checkpoint đã có sẵn: {zip_drive_file.name}, bỏ qua.")
            if split_type == "train": processed_train += 1
            else: processed_val += 1
            continue
            
        print(f"\n[*] [Đợt 2] Đang trích xuất: {m.name} ({m.size / (1024*1024):.1f} MB)...")
        tar.extract(m, temp_extract_dir)
        extracted_file = temp_extract_dir / m.name
        
        if not extracted_file.exists():
            continue
            
        ann_file = ann_file_map.get(actual_vid_stem.lower())
        frame_boxes = {}
        if ann_file and ann_file.exists():
            with open(ann_file, "r", encoding="utf-8") as f:
                for line in f:
                    p = line.strip().split(",")
                    if len(p) >= 6:
                        try:
                            fid = int(p[0])
                            x, y, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
                            if w > 5 and h > 10:
                                frame_boxes.setdefault(fid, []).append((x, y, w, h))
                        except ValueError:
                            continue
                            
        cap = cv2.VideoCapture(str(extracted_file))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        step = int(round(fps))
        
        temp_img_dir = Path(f"/content/tmp_{actual_vid_stem}")
        temp_img_dir.mkdir(parents=True, exist_ok=True)
        
        extracted_frames = 0
        f_idx = 0
        
        while True:
            r, frame = cap.read()
            if not r:
                break
            f_idx += 1
            
            if f_idx % step == 0 and f_idx in frame_boxes:
                h_img, w_img, _ = frame.shape
                frame_base = f"{actual_vid_stem}_f{f_idx:06d}"
                
                frame_resized = cv2.resize(frame, (1920, 1080))
                cv2.imwrite(str(temp_img_dir / f"{frame_base}.jpg"), frame_resized, [cv2.IMWRITE_JPEG_QUALITY, 85])
                
                yolo_lines = []
                for (x, y, w, h) in frame_boxes[f_idx]:
                    xc = max(0.0, min(1.0, (x + w / 2) / w_img))
                    yc = max(0.0, min(1.0, (y + h / 2) / h_img))
                    nw = max(0.0, min(1.0, w / w_img))
                    nh = max(0.0, min(1.0, h / h_img))
                    yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
                
                with open(temp_img_dir / f"{frame_base}.txt", "w", encoding="utf-8") as lf:
                    lf.write("\n".join(yolo_lines))
                
                extracted_frames += 1
                
        cap.release()
        if extracted_file.exists():
            os.remove(extracted_file)
        
        temp_zip = f"/content/{split_type}_{actual_vid_stem}.zip"
        with zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for f in temp_img_dir.glob("*.*"):
                zf.write(f, f.name)
                
        shutil.move(temp_zip, str(zip_drive_file))
        shutil.rmtree(temp_img_dir, ignore_errors=True)
        
        if split_type == "train": processed_train += 1
        else: processed_val += 1
        
        print(f"[+] [ĐỢT 2] ĐÃ LƯU CHECKPOINT DRIVE: {zip_drive_file.name} ({extracted_frames} ảnh)! ({processed_train}/{len(target_train)} Train | {processed_val}/{len(target_val)} Val)")
        
        if processed_train >= len(target_train) and processed_val >= len(target_val):
            print("\n🎉 ĐÃ HOÀN THÀNH TOÀN BỘ MỤC TIÊU ĐỢT 2!")
            break

shutil.rmtree(temp_extract_dir, ignore_errors=True)
print(f"\n🎉 KẾT THÚC ĐỢT 2! Đã bổ sung thành công vào Google Drive!")


### Bước 3.3: Bóc tách P-DESTRE ĐỢT 3 (Train 32..46 | Val 9..12)
> ⚡ Đợt 3 gồm 15 video Train + 4 video Val tiếp theo (~2.500 ảnh mới).

In [ ]:
# =====================================================================
# BƯỚC 3.3: BÓC TÁCH P-DESTRE ĐỢT 3 (Train 32..46 | Val 9..12)
# =====================================================================
import os
import tarfile
import cv2
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm
import random

tar_path = "/content/drive/MyDrive/human_tracking_drone/data/dataset.tar"
drive_output_dir = Path("/content/drive/MyDrive/human_tracking_drone/data/pdestre_processed")
drive_output_dir.mkdir(parents=True, exist_ok=True)

temp_extract_dir = Path("/content/temp_extract")
temp_extract_dir.mkdir(parents=True, exist_ok=True)

# 1. Đảm bảo file nhãn annotations đã sẵn sàng (trích xuất ~5 giây nếu chưa có)
ann_dir = Path("/content/pdestre_annotations")
if not ann_dir.exists() or len(list(ann_dir.glob('*.txt'))) == 0:
    print("[*] Đang trích xuất file nhãn annotations từ dataset.tar (rất nhẹ, ~5 giây)...")
    ann_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:*") as tar:
        for m in tar:
            if "annotation" in m.name.lower() and m.name.endswith(".txt") and not os.path.basename(m.name).startswith("._"):
                m.name = os.path.basename(m.name)
                tar.extract(m, ann_dir)
    print(f"[+] Đã trích xuất {len(list(ann_dir.glob('*.txt')))} file nhãn annotations!")
else:
    print(f"[i] Đã có sẵn {len(list(ann_dir.glob('*.txt')))} file nhãn annotations.")

# 2. Phân chia Train (80%) và Val (20%) cố định với seed 42
ann_files = sorted([f.stem for f in ann_dir.glob("*.txt")])
random.seed(42)
shuffled_vids = list(ann_files)
random.shuffle(shuffled_vids)

n_val = max(1, int(len(shuffled_vids) * 0.2))
val_vid_set = set(shuffled_vids[:n_val])
train_vid_set = set(shuffled_vids[n_val:])

train_vids_ordered = [v for v in shuffled_vids if v in train_vid_set]
val_vids_ordered = [v for v in shuffled_vids if v in val_vid_set]

START_TRAIN, END_TRAIN = 31, 46
START_VAL, END_VAL = 8, 12

target_train = set(train_vids_ordered[START_TRAIN:END_TRAIN])
target_val = set(val_vids_ordered[START_VAL:END_VAL])
all_targets = target_train | target_val
all_targets_map = {v.lower(): v for v in all_targets}
ann_file_map = {f.stem.lower(): f for f in ann_dir.glob('*.txt')}

print(f"[*] Mục tiêu ĐỢT 3: {len(target_train)} Train videos ({START_TRAIN+1}..{END_TRAIN}) + {len(target_val)} Val videos ({START_VAL+1}..{END_VAL})")

processed_train = 0
processed_val = 0

print("\n🚀 Bắt đầu Streaming xử lý Video ĐỢT 3...")
with tarfile.open(tar_path, "r:*") as tar:
    for m in tar:
        if not m.name.lower().endswith((".mp4", ".avi", ".mov")) or os.path.basename(m.name).startswith("._"):
            continue
            
        vid_stem = Path(m.name).stem
        if vid_stem.lower() not in all_targets_map:
            continue
            
        actual_vid_stem = all_targets_map[vid_stem.lower()]
        split_type = "train" if actual_vid_stem in target_train else "val"
        zip_drive_file = drive_output_dir / f"{split_type}_{actual_vid_stem}.zip"
        
        # Checkpoint: Bỏ qua nếu video đã có trên Drive
        if zip_drive_file.exists():
            print(f"[i] Checkpoint đã có sẵn: {zip_drive_file.name}, bỏ qua.")
            if split_type == "train": processed_train += 1
            else: processed_val += 1
            continue
            
        print(f"\n[*] [Đợt 3] Đang trích xuất: {m.name} ({m.size / (1024*1024):.1f} MB)...")
        tar.extract(m, temp_extract_dir)
        extracted_file = temp_extract_dir / m.name
        
        if not extracted_file.exists():
            continue
            
        ann_file = ann_file_map.get(actual_vid_stem.lower())
        frame_boxes = {}
        if ann_file and ann_file.exists():
            with open(ann_file, "r", encoding="utf-8") as f:
                for line in f:
                    p = line.strip().split(",")
                    if len(p) >= 6:
                        try:
                            fid = int(p[0])
                            x, y, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
                            if w > 5 and h > 10:
                                frame_boxes.setdefault(fid, []).append((x, y, w, h))
                        except ValueError:
                            continue
                            
        cap = cv2.VideoCapture(str(extracted_file))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        step = int(round(fps))
        
        temp_img_dir = Path(f"/content/tmp_{actual_vid_stem}")
        temp_img_dir.mkdir(parents=True, exist_ok=True)
        
        extracted_frames = 0
        f_idx = 0
        
        while True:
            r, frame = cap.read()
            if not r:
                break
            f_idx += 1
            
            if f_idx % step == 0 and f_idx in frame_boxes:
                h_img, w_img, _ = frame.shape
                frame_base = f"{actual_vid_stem}_f{f_idx:06d}"
                
                frame_resized = cv2.resize(frame, (1920, 1080))
                cv2.imwrite(str(temp_img_dir / f"{frame_base}.jpg"), frame_resized, [cv2.IMWRITE_JPEG_QUALITY, 85])
                
                yolo_lines = []
                for (x, y, w, h) in frame_boxes[f_idx]:
                    xc = max(0.0, min(1.0, (x + w / 2) / w_img))
                    yc = max(0.0, min(1.0, (y + h / 2) / h_img))
                    nw = max(0.0, min(1.0, w / w_img))
                    nh = max(0.0, min(1.0, h / h_img))
                    yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
                
                with open(temp_img_dir / f"{frame_base}.txt", "w", encoding="utf-8") as lf:
                    lf.write("\n".join(yolo_lines))
                
                extracted_frames += 1
                
        cap.release()
        if extracted_file.exists():
            os.remove(extracted_file)
        
        temp_zip = f"/content/{split_type}_{actual_vid_stem}.zip"
        with zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for f in temp_img_dir.glob("*.*"):
                zf.write(f, f.name)
                
        shutil.move(temp_zip, str(zip_drive_file))
        shutil.rmtree(temp_img_dir, ignore_errors=True)
        
        if split_type == "train": processed_train += 1
        else: processed_val += 1
        
        print(f"[+] [ĐỢT 3] ĐÃ LƯU CHECKPOINT DRIVE: {zip_drive_file.name} ({extracted_frames} ảnh)! ({processed_train}/{len(target_train)} Train | {processed_val}/{len(target_val)} Val)")
        
        if processed_train >= len(target_train) and processed_val >= len(target_val):
            print("\n🎉 ĐÃ HOÀN THÀNH TOÀN BỘ MỤC TIÊU ĐỢT 3!")
            break

shutil.rmtree(temp_extract_dir, ignore_errors=True)
print(f"\n🎉 KẾT THÚC ĐỢT 3! Đã bổ sung thành công vào Google Drive!")


### Bước 3.4: Bóc tách P-DESTRE ĐỢT 4 (Train 47..60 | Val 13..15 — Hoàn tất 100% P-DESTRE)
> 🎯 Đợt 4 gồm 14 video Train + 3 video Val cuối cùng (Phủ kín toàn bộ 75 video của P-DESTRE).

In [ ]:
# =====================================================================
# BƯỚC 3.4: BÓC TÁCH P-DESTRE ĐỢT 4 (Train 47..60 | Val 13..15)
# =====================================================================
import os
import tarfile
import cv2
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm
import random

tar_path = "/content/drive/MyDrive/human_tracking_drone/data/dataset.tar"
drive_output_dir = Path("/content/drive/MyDrive/human_tracking_drone/data/pdestre_processed")
drive_output_dir.mkdir(parents=True, exist_ok=True)

temp_extract_dir = Path("/content/temp_extract")
temp_extract_dir.mkdir(parents=True, exist_ok=True)

# 1. Đảm bảo file nhãn annotations đã sẵn sàng (trích xuất ~5 giây nếu chưa có)
ann_dir = Path("/content/pdestre_annotations")
if not ann_dir.exists() or len(list(ann_dir.glob('*.txt'))) == 0:
    print("[*] Đang trích xuất file nhãn annotations từ dataset.tar (rất nhẹ, ~5 giây)...")
    ann_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tar_path, "r:*") as tar:
        for m in tar:
            if "annotation" in m.name.lower() and m.name.endswith(".txt") and not os.path.basename(m.name).startswith("._"):
                m.name = os.path.basename(m.name)
                tar.extract(m, ann_dir)
    print(f"[+] Đã trích xuất {len(list(ann_dir.glob('*.txt')))} file nhãn annotations!")
else:
    print(f"[i] Đã có sẵn {len(list(ann_dir.glob('*.txt')))} file nhãn annotations.")

# 2. Phân chia Train (80%) và Val (20%) cố định với seed 42
ann_files = sorted([f.stem for f in ann_dir.glob("*.txt")])
random.seed(42)
shuffled_vids = list(ann_files)
random.shuffle(shuffled_vids)

n_val = max(1, int(len(shuffled_vids) * 0.2))
val_vid_set = set(shuffled_vids[:n_val])
train_vid_set = set(shuffled_vids[n_val:])

train_vids_ordered = [v for v in shuffled_vids if v in train_vid_set]
val_vids_ordered = [v for v in shuffled_vids if v in val_vid_set]

START_TRAIN, END_TRAIN = 46, 60
START_VAL, END_VAL = 12, 15

target_train = set(train_vids_ordered[START_TRAIN:END_TRAIN])
target_val = set(val_vids_ordered[START_VAL:END_VAL])
all_targets = target_train | target_val
all_targets_map = {v.lower(): v for v in all_targets}
ann_file_map = {f.stem.lower(): f for f in ann_dir.glob('*.txt')}

print(f"[*] Mục tiêu ĐỢT 4: {len(target_train)} Train videos ({START_TRAIN+1}..{END_TRAIN}) + {len(target_val)} Val videos ({START_VAL+1}..{END_VAL})")

processed_train = 0
processed_val = 0

print("\n🚀 Bắt đầu Streaming xử lý Video ĐỢT 4...")
with tarfile.open(tar_path, "r:*") as tar:
    for m in tar:
        if not m.name.lower().endswith((".mp4", ".avi", ".mov")) or os.path.basename(m.name).startswith("._"):
            continue
            
        vid_stem = Path(m.name).stem
        if vid_stem.lower() not in all_targets_map:
            continue
            
        actual_vid_stem = all_targets_map[vid_stem.lower()]
        split_type = "train" if actual_vid_stem in target_train else "val"
        zip_drive_file = drive_output_dir / f"{split_type}_{actual_vid_stem}.zip"
        
        # Checkpoint: Bỏ qua nếu video đã có trên Drive
        if zip_drive_file.exists():
            print(f"[i] Checkpoint đã có sẵn: {zip_drive_file.name}, bỏ qua.")
            if split_type == "train": processed_train += 1
            else: processed_val += 1
            continue
            
        print(f"\n[*] [Đợt 4] Đang trích xuất: {m.name} ({m.size / (1024*1024):.1f} MB)...")
        tar.extract(m, temp_extract_dir)
        extracted_file = temp_extract_dir / m.name
        
        if not extracted_file.exists():
            continue
            
        ann_file = ann_file_map.get(actual_vid_stem.lower())
        frame_boxes = {}
        if ann_file and ann_file.exists():
            with open(ann_file, "r", encoding="utf-8") as f:
                for line in f:
                    p = line.strip().split(",")
                    if len(p) >= 6:
                        try:
                            fid = int(p[0])
                            x, y, w, h = float(p[2]), float(p[3]), float(p[4]), float(p[5])
                            if w > 5 and h > 10:
                                frame_boxes.setdefault(fid, []).append((x, y, w, h))
                        except ValueError:
                            continue
                            
        cap = cv2.VideoCapture(str(extracted_file))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        step = int(round(fps))
        
        temp_img_dir = Path(f"/content/tmp_{actual_vid_stem}")
        temp_img_dir.mkdir(parents=True, exist_ok=True)
        
        extracted_frames = 0
        f_idx = 0
        
        while True:
            r, frame = cap.read()
            if not r:
                break
            f_idx += 1
            
            if f_idx % step == 0 and f_idx in frame_boxes:
                h_img, w_img, _ = frame.shape
                frame_base = f"{actual_vid_stem}_f{f_idx:06d}"
                
                frame_resized = cv2.resize(frame, (1920, 1080))
                cv2.imwrite(str(temp_img_dir / f"{frame_base}.jpg"), frame_resized, [cv2.IMWRITE_JPEG_QUALITY, 85])
                
                yolo_lines = []
                for (x, y, w, h) in frame_boxes[f_idx]:
                    xc = max(0.0, min(1.0, (x + w / 2) / w_img))
                    yc = max(0.0, min(1.0, (y + h / 2) / h_img))
                    nw = max(0.0, min(1.0, w / w_img))
                    nh = max(0.0, min(1.0, h / h_img))
                    yolo_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")
                
                with open(temp_img_dir / f"{frame_base}.txt", "w", encoding="utf-8") as lf:
                    lf.write("\n".join(yolo_lines))
                
                extracted_frames += 1
                
        cap.release()
        if extracted_file.exists():
            os.remove(extracted_file)
        
        temp_zip = f"/content/{split_type}_{actual_vid_stem}.zip"
        with zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for f in temp_img_dir.glob("*.*"):
                zf.write(f, f.name)
                
        shutil.move(temp_zip, str(zip_drive_file))
        shutil.rmtree(temp_img_dir, ignore_errors=True)
        
        if split_type == "train": processed_train += 1
        else: processed_val += 1
        
        print(f"[+] [ĐỢT 4] ĐÃ LƯU CHECKPOINT DRIVE: {zip_drive_file.name} ({extracted_frames} ảnh)! ({processed_train}/{len(target_train)} Train | {processed_val}/{len(target_val)} Val)")
        
        if processed_train >= len(target_train) and processed_val >= len(target_val):
            print("\n🎉 ĐÃ HOÀN THÀNH TOÀN BỘ MỤC TIÊU ĐỢT 4!")
            break

shutil.rmtree(temp_extract_dir, ignore_errors=True)
print(f"\n🎉 KẾT THÚC ĐỢT 4! Đã bổ sung thành công vào Google Drive!")


### Bước 4: Tự động Giải nén Toàn bộ Checkpoints & Hợp nhất với VisDrone (Mất ~10 giây)
> Ô này sẽ tự động phát hiện và giải nén **tất cả** các file zip checkpoint đang có trên Google Drive (dù bạn chạy 1 đợt hay cả 4 đợt) và hợp nhất hoàn hảo cùng 3.299 ảnh của VisDrone.

In [ ]:
import os
import zipfile
from pathlib import Path
from tqdm import tqdm

pdestre_ssd = Path("/content/dataset/pdestre_yolo")
os.makedirs(pdestre_ssd / "images/train", exist_ok=True)
os.makedirs(pdestre_ssd / "images/val", exist_ok=True)
os.makedirs(pdestre_ssd / "labels/train", exist_ok=True)
os.makedirs(pdestre_ssd / "labels/val", exist_ok=True)

drive_checkpoints = list(Path("/content/drive/MyDrive/human_tracking_drone/data/pdestre_processed").glob("*.zip"))
print(f"[*] Đang giải nén {len(drive_checkpoints)} checkpoint từ Google Drive vào SSD Colab...")

for czip in tqdm(drive_checkpoints):
    split = "train" if czip.name.startswith("train_") else "val"
    with zipfile.ZipFile(czip, 'r') as zf:
        for name in zf.namelist():
            if name.endswith(".jpg"):
                zf.extract(name, pdestre_ssd / f"images/{split}")
            elif name.endswith(".txt"):
                zf.extract(name, pdestre_ssd / f"labels/{split}")

# Giải nén VisDrone nếu chưa giải nén
visdrone_ssd = Path("/content/dataset/visdrone_filtered")
if not (visdrone_ssd / "images/train").exists():
    print("[*] Đang giải nén VisDrone vào SSD...")
    with zipfile.ZipFile("/content/drive/MyDrive/human_tracking_drone/data/visdrone_filtered_yolo.zip", 'r') as zf:
        zf.extractall(visdrone_ssd)

# Lập danh sách ảnh Train hợp nhất
visdrone_train = list((visdrone_ssd / "images/train").glob("*.jpg"))
pdestre_train = list((pdestre_ssd / "images/train").glob("*.jpg"))
pdestre_val = list((pdestre_ssd / "images/val").glob("*.jpg"))

all_train = [str(p) for p in visdrone_train + pdestre_train]
with open("/content/dataset/train_merged.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(all_train) + "\n")

with open("/content/dataset/pdestre_val.txt", "w", encoding="utf-8") as f:
    f.write("\n".join([str(p) for p in pdestre_val]) + "\n")

# Tạo file YAML cấu hình huấn luyện hợp nhất
yaml_merged = """
path: /content/dataset
train: /content/dataset/train_merged.txt
val: visdrone_filtered/images/val

names:
  0: person
"""
with open("/content/dataset/merged_drone_dataset.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_merged.strip())

print(f"\n🎉 Hợp nhất thành công: {len(all_train)} ảnh Train ({len(visdrone_train)} VisDrone + {len(pdestre_train)} P-DESTRE)!")
print(f"   Val set giữ nguyên VisDrone Val (440 ảnh) để so sánh đối chứng khoa học với mốc 57.26%!")

### Bước 5: Huấn luyện YOLO26n 80 Epochs (Mục tiêu: $mAP_{50} \ge 60\%$)
> Khởi tạo từ trọng số `best.pt` của VisDrone (57.26%) để kế thừa đặc trưng, tối ưu hóa với Cosine LR và Mosaic shutdown.

In [ ]:
from ultralytics import YOLO
import os

run1_best = "/content/drive/MyDrive/human_tracking_drone/models/yolo26n_visdrone_filtered_run1/weights/best.pt"
if os.path.exists(run1_best):
    print(f"[*] Khởi tạo từ checkpoint tốt nhất của VisDrone: {run1_best}")
    model = YOLO(run1_best)
else:
    print("[*] Checkpoint VisDrone chưa có, khởi tạo yolo26n.pt gốc")
    model = YOLO("yolo26n.pt")

# Bắt đầu huấn luyện 80 epochs
results = model.train(
    data="/content/dataset/merged_drone_dataset.yaml",
    epochs=80,
    imgsz=640,
    batch=32,
    device=0,
    cos_lr=True,
    close_mosaic=10,
    project="/content/drive/MyDrive/human_tracking_drone/models",
    name="yolo26n_merged_visdrone_pdestre_80epochs",
    save=True
)

print("[+] Huấn luyện 80 epochs hoàn tất! Checkpoint best.pt đã lưu vào Google Drive.")

### Bước 6: Đánh giá Đối chứng Đa miền Chuẩn hóa (Toàn bộ 15 Video Val P-DESTRE + VisDrone Val)
> 🎯 **Ô độc lập hoàn toàn (Self-contained):** Tự động cài đặt `ultralytics` và kết nối Google Drive nếu chưa có, giải nén đủ 15 video Val của P-DESTRE và 440 ảnh VisDrone Val để đo đạc đối chứng công bằng giữa các phiên bản mô hình (YOLO26n vs YOLO26s).

In [ ]:
# =====================================================================
# BƯỚC 6: ĐÁNH GIÁ ĐA CHIỀU NCKH — TOÀN BỘ 15 VIDEO VAL P-DESTRE & VISDRONE
# Tự động cài đặt ultralytics và mount Drive nếu chạy độc lập
# =====================================================================
import os
import sys
import shutil
import zipfile
from pathlib import Path

# 0. Tự động cài đặt ultralytics nếu môi trường Colab mới khởi tạo
try:
    from ultralytics import YOLO
except ImportError:
    print('[*] Chưa tìm thấy thư viện ultralytics, đang tự động cài đặt...')
    os.system('pip install -q "ultralytics>=8.3.0" opencv-python tqdm')
    from ultralytics import YOLO

# Tự động mount Google Drive nếu chưa mount
if not os.path.exists('/content/drive/MyDrive'):
    print('[*] Đang kết nối Google Drive...')
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f'[!] Lưu ý kết nối Drive: {e}')

import torch

dataset_dir = Path('/content/dataset')
visdrone_val_dir = dataset_dir / 'visdrone_filtered/images/val'
pdestre_val_dir = dataset_dir / 'pdestre_yolo/images/val'
pdestre_lbl_dir = dataset_dir / 'pdestre_yolo/labels/val'

drive_base = Path('/content/drive/MyDrive/human_tracking_drone')
drive_data_dir = drive_base / 'data'
drive_pdestre_dir = drive_data_dir / 'pdestre_processed'

device = 0 if torch.cuda.is_available() else 'cpu'
batch_size = 16 if device == 0 else 4

print('=' * 75)
print(f'🎯 BẮT ĐẦU QUY TRÌNH KIỂM THỬ ĐỐI CHỨNG ĐA MIỀN (Thiết bị: {device})')
print('=' * 75)

# 1. Giải nén VisDrone Val (440 ảnh sạch)
if not visdrone_val_dir.exists() or len(list(visdrone_val_dir.glob('*.jpg'))) < 400:
    print('[1/2] Đang giải nén tập VisDrone Val (440 ảnh) vào SSD...')
    visdrone_zip = drive_data_dir / 'visdrone_filtered_yolo.zip'
    if visdrone_zip.exists():
        with zipfile.ZipFile(visdrone_zip, 'r') as zf:
            zf.extractall(dataset_dir / 'visdrone_filtered')
n_visdrone = len(list(visdrone_val_dir.glob('*.jpg')))
print(f'  [+] VisDrone Val: {n_visdrone:,} ảnh sẵn sàng.')

# 2. Giải nén TOÀN BỘ 15 video Val của P-DESTRE (đảm bảo đầy đủ không bị thiếu)
val_zips = sorted(list(drive_pdestre_dir.glob('val_*.zip')))
print(f'[2/2] Tìm thấy {len(val_zips)} gói zip video Val P-DESTRE trên Drive...')

pdestre_val_dir.mkdir(parents=True, exist_ok=True)
pdestre_lbl_dir.mkdir(parents=True, exist_ok=True)

print('  [*] Đang trích xuất toàn bộ 15 chuỗi video Val vào SSD...')
for vz in val_zips:
    with zipfile.ZipFile(vz, 'r') as zf:
        for name in zf.namelist():
            fname = Path(name).name
            if not fname:
                continue
            if name.endswith('.jpg'):
                zf.extract(name, '/content/temp_val')
                shutil.move(f'/content/temp_val/{name}', pdestre_val_dir / fname)
            elif name.endswith('.txt'):
                zf.extract(name, '/content/temp_val')
                shutil.move(f'/content/temp_val/{name}', pdestre_lbl_dir / fname)
shutil.rmtree('/content/temp_val', ignore_errors=True)

n_pdestre_imgs = len(list(pdestre_val_dir.glob('*.jpg')))
print(f'  [+] P-DESTRE Val: {n_pdestre_imgs:,} ảnh từ {len(val_zips)} chuỗi video sẵn sàng.')

# 3. Tạo 2 file YAML kiểm định độc lập (Ultralytics yêu cầu bắt buộc cả 2 khóa train và val)
yaml_visdrone = f'''path: {dataset_dir / 'visdrone_filtered'}
train: images/val
val: images/val
names:
  0: person
'''
with open('/content/dataset/val_visdrone.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_visdrone.strip())

yaml_pdestre = f'''path: {dataset_dir / 'pdestre_yolo'}
train: images/val
val: images/val
names:
  0: person
'''
with open('/content/dataset/val_pdestre_15vids.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_pdestre.strip())

# 4. Xác định danh sách các mô hình cần đánh giá trên Drive
candidate_models = {
    'YOLO26n Baseline': [
        drive_base / 'models/yolo26n_visdrone/weights/best.pt',
        drive_base / 'models/yolo26n_visdrone_filtered_run1/weights/best.pt',
    ],
    'YOLO26n Merged': [
        drive_base / 'models/yolo26n_merged/weights/best.pt',
        drive_base / 'models/yolo26n_merged_run2/weights/best.pt',
        drive_base / 'models/yolo26n_merged_visdrone_pdestre_80epochs/weights/best.pt',
    ],
    'YOLO26s Merged (Trạm mặt đất)': [
        drive_base / 'models/yolo26s_merged/weights/best.pt',
        drive_base / 'models/yolo26s_merged_run3/weights/best.pt',
    ]
}

models_to_test = {}
for label, paths in candidate_models.items():
    for p in paths:
        if p.exists():
            models_to_test[label] = p
            break

print('\n[*] Danh sách mô hình tìm thấy trên Drive:')
for lbl, p in models_to_test.items():
    print(f'  • {lbl:<30} -> {p.name}')

# 5. Tiến hành đo đạc đối chứng
results_summary = []

for label, weight_path in models_to_test.items():
    print('\n' + '=' * 70)
    print(f'🔄 ĐANG ĐÁNH GIÁ: {label}')
    print('=' * 70)
    m = YOLO(str(weight_path))
    
    # VisDrone Val
    print(f'[*] [1/2] Đánh giá trên VisDrone Val ({n_visdrone} ảnh)...')
    res_vd = m.val(data='/content/dataset/val_visdrone.yaml', split='val', imgsz=640, batch=batch_size, device=device, plots=False, verbose=False)
    
    # P-DESTRE Val (15 video)
    print(f'[*] [2/2] Đánh giá trên P-DESTRE Val ({len(val_zips)} chuỗi video 4K)...')
    res_pd = m.val(data='/content/dataset/val_pdestre_15vids.yaml', split='val', imgsz=640, batch=batch_size, device=device, plots=False, verbose=False)
    
    results_summary.append({
        'Model': label,
        'VD_mAP50': res_vd.box.map50 * 100,
        'VD_mAP': res_vd.box.map * 100,
        'VD_P': res_vd.box.mp * 100,
        'VD_R': res_vd.box.mr * 100,
        'PD_mAP50': res_pd.box.map50 * 100,
        'PD_mAP': res_pd.box.map * 100,
        'PD_P': res_pd.box.mp * 100,
        'PD_R': res_pd.box.mr * 100,
        'Inf_ms': res_vd.speed['inference']
    })

# 6. In Bảng Tổng Hợp Đối Chứng Chuẩn Mực
print('\n' + '=' * 105)
print('📊 BẢNG TỔNG HỢP THỰC NGHIỆM ĐỐI CHỨNG ĐA MIỀN (CHUẨN HÓA CÙNG ĐIỀU KIỆN)')
print('=' * 105)
header = f"{'Mô Hình':<30} | {'VD mAP50':<10} | {'VD Recall':<10} | {'PD 15vids mAP50':<16} | {'PD 15vids Rec':<14} | {'Inference'}"
print(header)
print('-' * 105)

for r in results_summary:
    line = f"{r['Model']:<30} | {r['VD_mAP50']:>8.2f}% | {r['VD_R']:>8.2f}% | {r['PD_mAP50']:>14.2f}% | {r['PD_R']:>12.2f}% | {r['Inf_ms']:>6.2f} ms"
    print(line)
print('=' * 105)

# Lưu kết quả về Google Drive
out_txt = drive_base / 'models/benchmark_15vids_comparison.txt'
try:
    with open(out_txt, 'w', encoding='utf-8') as f:
        f.write('BẢNG TỔNG HỢP THỰC NGHIỆM ĐỐI CHỨNG ĐA MIỀN (15 VIDEO VAL P-DESTRE + VISDRONE VAL)\n')
        f.write(header + '\n' + '-' * 105 + '\n')
        for r in results_summary:
            f.write(f"{r['Model']:<30} | {r['VD_mAP50']:>8.2f}% | {r['VD_R']:>8.2f}% | {r['PD_mAP50']:>14.2f}% | {r['PD_R']:>12.2f}% | {r['Inf_ms']:>6.2f} ms\n")
    print(f'\n[+] Đã lưu bảng kết quả vào: {out_txt}')
except Exception as e:
    print(f'[!] Không thể ghi file ra Drive: {e}')
